# Etapa 2 — Pré-processamento e construção das referências

Esta etapa prepara os datasets para o processamento estatístico. Ela descompacta os arquivos em `Extraidos/`, remove dados descartáveis e cria `output/referencias_processamento.json`.

O índice contém apenas referências relativas aos arquivos físicos que passaram pelos filtros. Ele não copia assessments, códigos ou logs. Por padrão, o recorte seleciona somente assessments do tipo **`exam`**.


In [1]:
from pathlib import Path

# O notebook deve ser executado com o diretório de trabalho em Etapa_2.
# Estes são os únicos caminhos globais usados pelo pré-processamento.
RAIZ_PROJETO = Path.cwd().parent
PASTA_DATASETS = RAIZ_PROJETO / 'DataSets'
PASTA_EXTRAIDOS = RAIZ_PROJETO / 'Extraidos'
PASTA_SAIDA = Path.cwd() / 'output'
CAMINHO_REFERENCIAS = PASTA_SAIDA / 'referencias_processamento.json'
CAMINHO_CONTROLE_DESCOMPACTACAO = PASTA_SAIDA / 'controle_descompactacao.json'

# Filtros aplicados durante a construção do índice.
# Valores aceitos: 'exam', 'homework' ou 'todos'.
TIPO_ASSESSMENT_DESEJADO = 'exam'
NOME_CURSO_DESEJADO = 'todos'

print(f'Datasets: {PASTA_DATASETS.resolve()}')
print(f'Extraidos: {PASTA_EXTRAIDOS.resolve()}')
print(f'Referências: {CAMINHO_REFERENCIAS.resolve()}')
print(f'Controle de descompactação: {CAMINHO_CONTROLE_DESCOMPACTACAO.resolve()}')


Datasets: D:\Trabalho_Super\QuestionInsight\DataSets
Extraidos: D:\Trabalho_Super\QuestionInsight\Extraidos
Referências: D:\Trabalho_Super\QuestionInsight\Etapa_2\output\referencias_processamento.json
Controle de descompactação: D:\Trabalho_Super\QuestionInsight\Etapa_2\output\controle_descompactacao.json


## 1. Entradas, controle de descompactação e parâmetros

Os datasets `.tar.gz` são lidos de `DataSets/` e extraídos para `Extraidos/`. O arquivo `output/controle_descompactacao.json` registra o status de cada dataset. Se esse controle não existir, ele é criado automaticamente com todos os datasets marcados como pendentes.

Os parâmetros principais são `TIPO_ASSESSMENT_DESEJADO`, que aceita `exam`, `homework` ou `todos`, e `NOME_CURSO_DESEJADO`, que permite restringir o curso. A execução também valida a quantidade e o tamanho dos arquivos extraídos quando esses metadados estão disponíveis.


In [2]:
import json
import shutil
import tarfile
from datetime import datetime, timezone


def formatar_tamanho(numero_bytes: int) -> str:
    tamanho = float(numero_bytes)
    for unidade in ('B', 'KB', 'MB', 'GB', 'TB'):
        if tamanho < 1024 or unidade == 'TB':
            return f'{tamanho:.2f} {unidade}'
        tamanho /= 1024
    return f'{numero_bytes} B'


def medir_arquivos_regulares(pasta: Path) -> tuple[int, int]:
    """Retorna quantidade e tamanho dos arquivos presentes na pasta."""
    quantidade = 0
    tamanho_total = 0
    for caminho in pasta.rglob('*'):
        if caminho.is_file():
            quantidade += 1
            tamanho_total += caminho.stat().st_size
    return quantidade, tamanho_total


def obter_metadados_tar(arquivo: Path) -> tuple[int, int]:
    """Calcula quantidade e tamanho esperados dos arquivos do TAR."""
    with tarfile.open(arquivo, 'r:gz') as arquivo_tar:
        membros = arquivo_tar.getmembers()
    arquivos_regulares = [membro for membro in membros if membro.isfile()]
    return len(arquivos_regulares), sum(membro.size for membro in arquivos_regulares)


def criar_controle_inicial() -> dict:
    """Cria um controle pendente a partir dos TARs existentes no DataSets."""
    arquivos = sorted(PASTA_DATASETS.rglob('*.tar.gz'))
    datasets = []
    for arquivo in arquivos:
        nome = arquivo.name.removesuffix('.tar.gz')
        periodo = nome.removeprefix('cb_dataset_').split('_v', 1)[0]
        datasets.append({
            'nome': nome,
            'periodo': periodo,
            'status': 'pendente',
            'marcador_deve_existir': True,
            'motivo': 'Controle criado automaticamente porque o arquivo não existia.',
            'arquivo_origem': arquivo.name,
        })
    controle = {
        'schema_version': 1,
        'descricao': 'Controle central de descompactação criado automaticamente.',
        'regra': 'Dataset só é considerado concluído após extração e validação.',
        'datasets': datasets,
        'resumo': {
            'total_datasets': len(datasets),
            'datasets_pendentes_reextracao': len(datasets),
            'datasets_concluidos': 0,
            'periodos_pendentes': sorted({item['periodo'].split('_', 1)[0] for item in datasets}),
            'periodos_concluidos': [],
        },
        'uso': {
            'pendentes': 'Serão extraídos e validados na execução atual.',
            'concluidos': 'Não há datasets concluídos neste controle inicial.',
        },
    }
    CAMINHO_CONTROLE_DESCOMPACTACAO.parent.mkdir(parents=True, exist_ok=True)
    CAMINHO_CONTROLE_DESCOMPACTACAO.write_text(
        json.dumps(controle, ensure_ascii=False, indent=2) + '\n',
        encoding='utf-8',
    )
    print(f'Controle central criado automaticamente: {CAMINHO_CONTROLE_DESCOMPACTACAO}')
    return controle


def carregar_controle() -> dict:
    """Carrega o controle central ou cria um inicial quando ele não existe."""
    if not CAMINHO_CONTROLE_DESCOMPACTACAO.is_file():
        return criar_controle_inicial()
    try:
        return json.loads(CAMINHO_CONTROLE_DESCOMPACTACAO.read_text(encoding='utf-8'))
    except (json.JSONDecodeError, OSError) as exc:
        raise RuntimeError(
            f'Controle central inválido ou ilegível: {CAMINHO_CONTROLE_DESCOMPACTACAO}'
        ) from exc


def salvar_controle(controle: dict) -> None:
    """Atualiza o JSON central de forma atômica."""
    temporario = CAMINHO_CONTROLE_DESCOMPACTACAO.with_suffix('.json.tmp')
    temporario.write_text(
        json.dumps(controle, ensure_ascii=False, indent=2) + '\n',
        encoding='utf-8',
    )
    temporario.replace(CAMINHO_CONTROLE_DESCOMPACTACAO)


def registro_do_dataset(controle: dict, nome_dataset: str) -> dict:
    for registro in controle.get('datasets', []):
        if registro.get('nome') == nome_dataset:
            return registro
    raise KeyError(
        f'Dataset não cadastrado no controle central: {nome_dataset}'
    )


def extracao_esta_valida(
    registro: dict,
    arquivo: Path,
    pasta_destino: Path,
    quantidade_esperada: int,
    tamanho_esperado: int,
) -> bool:
    """Valida status central, origem e quantidade/tamanho extraídos."""
    if registro.get('status') != 'concluido' or not pasta_destino.is_dir():
        return False
    # Registros concluídos do controle inicial podem não conter os
    # metadados de tamanho. Nesse caso, preservamos o processamento anterior
    # e não reextraímos uma pasta existente só por falta desses campos.
    metadados_disponiveis = all(
        campo in registro
        for campo in (
            'arquivo_origem',
            'tamanho_comprimido_bytes',
            'quantidade_arquivos',
            'tamanho_descompactado_bytes',
        )
    )
    if not metadados_disponiveis:
        return pasta_destino.is_dir()
    if registro.get('arquivo_origem') != arquivo.name:
        return False
    if registro.get('tamanho_comprimido_bytes') != arquivo.stat().st_size:
        return False
    if registro.get('quantidade_arquivos') != quantidade_esperada:
        return False
    if registro.get('tamanho_descompactado_bytes') != tamanho_esperado:
        return False
    quantidade_atual, tamanho_atual = medir_arquivos_regulares(pasta_destino)
    return quantidade_atual == quantidade_esperada and tamanho_atual == tamanho_esperado


def atualizar_registro_concluido(
    registro: dict,
    arquivo: Path,
    quantidade_arquivos: int,
    tamanho_descompactado: int,
) -> None:
    registro.update({
        'status': 'concluido',
        'marcador_deve_existir': False,
        'arquivo_origem': arquivo.name,
        'tamanho_comprimido_bytes': arquivo.stat().st_size,
        'quantidade_arquivos': quantidade_arquivos,
        'tamanho_descompactado_bytes': tamanho_descompactado,
        'concluido_em': datetime.now(timezone.utc).isoformat(),
    })


def descompactar_tar_gz(
    arquivo: Path,
    pasta_destino: Path,
    registro: dict,
    controle: dict,
) -> bool:
    """Ignora somente quando o JSON central comprova uma extração completa."""
    quantidade_esperada, tamanho_esperado = obter_metadados_tar(arquivo)

    if extracao_esta_valida(
        registro,
        arquivo,
        pasta_destino,
        quantidade_esperada,
        tamanho_esperado,
    ):
        print(
            f'Validação central OK: {arquivo.name} | '
            f'{quantidade_esperada:,} arquivos | {formatar_tamanho(tamanho_esperado)}'
        )
        return False

    if pasta_destino.exists():
        print(f'Extração ausente ou incompleta; refazendo: {pasta_destino}')
        shutil.rmtree(pasta_destino)
    pasta_destino.mkdir(parents=True, exist_ok=True)

    try:
        with tarfile.open(arquivo, 'r:gz') as arquivo_tar:
            arquivo_tar.extractall(path=pasta_destino)

        quantidade_atual, tamanho_atual = medir_arquivos_regulares(pasta_destino)
        if quantidade_atual != quantidade_esperada or tamanho_atual != tamanho_esperado:
            raise RuntimeError(
                f'Validação pós-extração falhou para {arquivo.name}: '
                f'esperado {quantidade_esperada} arquivos/{tamanho_esperado} bytes, '
                f'encontrado {quantidade_atual} arquivos/{tamanho_atual} bytes.'
            )

        atualizar_registro_concluido(
            registro,
            arquivo,
            quantidade_esperada,
            tamanho_esperado,
        )
        salvar_controle(controle)
        print(
            f'Extração validada e controle atualizado: {arquivo.name} | '
            f'{quantidade_atual:,} arquivos | {formatar_tamanho(tamanho_atual)}'
        )
        return True
    except Exception:
        shutil.rmtree(pasta_destino, ignore_errors=True)
        raise


PASTA_EXTRAIDOS.mkdir(parents=True, exist_ok=True)
CONTROLE_DESCOMPACTACAO = carregar_controle()
ARQUIVOS_TAR_GZ = sorted(PASTA_DATASETS.rglob('*.tar.gz'))
ARQUIVOS_POR_NOME = {arquivo.name.removesuffix('.tar.gz'): arquivo for arquivo in ARQUIVOS_TAR_GZ}

# O JSON central é a fonte de verdade para decidir o que ainda falta.
registros = CONTROLE_DESCOMPACTACAO.get('datasets', [])
nomes_encontrados = set(ARQUIVOS_POR_NOME)
registros_concluidos = [registro for registro in registros if registro.get('status') == 'concluido']
registros_pendentes = [registro for registro in registros if registro.get('status') != 'concluido']
registros_sem_arquivo = [registro for registro in registros if registro.get('nome') not in nomes_encontrados]

print(f'Pacotes encontrados no DataSets: {len(ARQUIVOS_TAR_GZ):,}')
print(f'Pacotes concluídos segundo o controle central: {len(registros_concluidos):,}')
print(f'Pacotes pendentes segundo o controle central: {len(registros_pendentes):,}')

print('\nPacotes já concluídos:')
if registros_concluidos:
    for indice_concluido, registro in enumerate(registros_concluidos, start=1):
        print(f' - [{indice_concluido}/{len(registros_concluidos)} concluídos] {registro.get("nome")}')
else:
    print(' - Nenhum pacote concluído.')

if registros_sem_arquivo:
    print('Atenção: há registros no JSON sem arquivo .tar.gz correspondente:')
    for registro in registros_sem_arquivo:
        print(f" - {registro.get('nome')}")

if registros_pendentes:
    print('Pacotes que ainda faltam:')
    for registro in registros_pendentes:
        nome = registro.get('nome')
        print(f" - {nome} ({registro.get('status', 'sem_status')})")
else:
    print('Nenhum pacote pendente no controle central.')

pacotes_processados_agora = 0
for indice_pendente, registro in enumerate(registros_pendentes, start=1):
    nome_dataset = registro.get('nome')
    arquivo = ARQUIVOS_POR_NOME.get(nome_dataset)
    if arquivo is None:
        print(f'[{indice_pendente}/{len(registros_pendentes)} pendentes] Ausente no DataSets: {nome_dataset}')
        continue

    pasta_destino = PASTA_EXTRAIDOS / nome_dataset
    print(f'[{indice_pendente}/{len(registros_pendentes)} pendentes] Processando: {nome_dataset}')
    foi_descompactado = descompactar_tar_gz(
        arquivo,
        pasta_destino,
        registro,
        CONTROLE_DESCOMPACTACAO,
    )
    pacotes_processados_agora += int(foi_descompactado)
    situacao = 'Descompactado' if foi_descompactado else 'Já validado'
    print(
        f'[{indice_pendente}/{len(registros_pendentes)} pendentes] '
        f'{situacao}: {arquivo.name}'
    )

print()
print('Processamento concluído.')
print(f'Pacotes totais encontrados: {len(ARQUIVOS_TAR_GZ):,}')
print(f'Pacotes já concluídos antes desta execução: {len(registros_concluidos):,}')
print(f'Pacotes pendentes processados nesta execução: {pacotes_processados_agora:,}')
print(f'Pacotes restantes após esta execução: {sum(registro.get("status") != "concluido" for registro in CONTROLE_DESCOMPACTACAO.get("datasets", [])):,}')


Controle central criado automaticamente: D:\Trabalho_Super\QuestionInsight\Etapa_2\output\controle_descompactacao.json
Pacotes encontrados no DataSets: 15
Pacotes concluídos segundo o controle central: 0
Pacotes pendentes segundo o controle central: 15

Pacotes já concluídos:
 - Nenhum pacote concluído.
Pacotes que ainda faltam:
 - cb_dataset_2016_1_v1.81 (pendente)
 - cb_dataset_2016_2_v1.81 (pendente)
 - cb_dataset_2017_1_v1.81 (pendente)
 - cb_dataset_2017_2_v1.81 (pendente)
 - cb_dataset_2018_1_v1.81 (pendente)
 - cb_dataset_2018_2_v1.81 (pendente)
 - cb_dataset_2019_1_v1.81 (pendente)
 - cb_dataset_2019_2_v1.81 (pendente)
 - cb_dataset_2021_1_v1.81 (pendente)
 - cb_dataset_2021_2_v1.81 (pendente)
 - cb_dataset_2022_1_v1.81 (pendente)
 - cb_dataset_2022_2_v1.81 (pendente)
 - cb_dataset_2023_1_v1.81 (pendente)
 - cb_dataset_2023_2_v1.81 (pendente)
 - cb_dataset_2024_1_v1.81 (pendente)
[1/15 pendentes] Processando: cb_dataset_2016_1_v1.81


Extração ausente ou incompleta; refazendo: D:\Trabalho_Super\QuestionInsight\Extraidos\cb_dataset_2016_1_v1.81


Extração validada e controle atualizado: cb_dataset_2016_1_v1.81.tar.gz | 96,880 arquivos | 1.58 GB
[1/15 pendentes] Descompactado: cb_dataset_2016_1_v1.81.tar.gz
[2/15 pendentes] Processando: cb_dataset_2016_2_v1.81


Extração ausente ou incompleta; refazendo: D:\Trabalho_Super\QuestionInsight\Extraidos\cb_dataset_2016_2_v1.81


Extração validada e controle atualizado: cb_dataset_2016_2_v1.81.tar.gz | 30,685 arquivos | 547.03 MB
[2/15 pendentes] Descompactado: cb_dataset_2016_2_v1.81.tar.gz
[3/15 pendentes] Processando: cb_dataset_2017_1_v1.81


Extração ausente ou incompleta; refazendo: D:\Trabalho_Super\QuestionInsight\Extraidos\cb_dataset_2017_1_v1.81


... (21 outputs left out)

Extração ausente ou incompleta; refazendo: D:\Trabalho_Super\QuestionInsight\Extraidos\cb_dataset_2023_2_v1.81


Extração validada e controle atualizado: cb_dataset_2023_2_v1.81.tar.gz | 72,078 arquivos | 2.06 GB
[14/15 pendentes] Descompactado: cb_dataset_2023_2_v1.81.tar.gz
[15/15 pendentes] Processando: cb_dataset_2024_1_v1.81


Extração ausente ou incompleta; refazendo: D:\Trabalho_Super\QuestionInsight\Extraidos\cb_dataset_2024_1_v1.81


Extração validada e controle atualizado: cb_dataset_2024_1_v1.81.tar.gz | 86,085 arquivos | 2.51 GB
[15/15 pendentes] Descompactado: cb_dataset_2024_1_v1.81.tar.gz

Processamento concluído.
Pacotes totais encontrados: 15
Pacotes já concluídos antes desta execução: 0
Pacotes pendentes processados nesta execução: 15
Pacotes restantes após esta execução: 0


## 2. Descoberta, seleção e validação das referências

O notebook percorre os arquivos de assessment e lê seus metadados. Um assessment só entra no índice quando atende ao tipo e ao curso selecionados. As questões são associadas ao assessment pai, preservando o escopo do dataset e do período.

Depois de conhecer os assessments selecionados, os `execution logs` candidatos são validados em paralelo. Um execution só é aprovado quando contém o último `SUBMITION` com `GRADE`, não contém `ERROR` nesse bloco e possui código Python sintaticamente válido. Somente então são montadas as referências de `executions`, `codes` e `codemirror` associadas a assessments e questões selecionados. As rejeições são registradas em `output/execucoes_rejeitadas_etapa2.json`.


In [3]:
import ast
import json
import os
import re
import unicodedata
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime

try:
    from tqdm import tqdm
except ImportError:
    def tqdm(iterable=None, total=None, desc=None, **_kwargs):
        if iterable is None:
            return range(total or 0)
        return iterable


ETAPA2_WORKERS = 16
ETAPA2_LOG_PATH = PASTA_SAIDA / 'etapa2_progresso.log'


def registrar_progresso(mensagem: str) -> None:
    '''Escreve o andamento no output da célula e em um log persistente.'''
    linha = f'[{datetime.now().strftime("%Y-%m-%d %H:%M:%S")}] {mensagem}'
    print(linha, flush=True)
    ETAPA2_LOG_PATH.parent.mkdir(parents=True, exist_ok=True)
    with ETAPA2_LOG_PATH.open('a', encoding='utf-8') as arquivo_log:
        arquivo_log.write(linha + '\n')
        arquivo_log.flush()




def normalizar(texto: str) -> str:
    sem_acento = ''.join(
        caractere
        for caractere in unicodedata.normalize('NFD', str(texto))
        if unicodedata.category(caractere) != 'Mn'
    )
    return sem_acento.strip().lower()


def ler_assessment_file(filepath: Path) -> dict[str, object]:
    dados: dict[str, object] = {'question_ids': []}
    em_exercicios = False
    try:
        for raw_line in filepath.read_text(encoding='utf-8').splitlines():
            line = raw_line.strip()
            if line.startswith('-- EXERCISES:'):
                em_exercicios = True
                continue
            if line.startswith('-- ASSESSMENT DATA:'):
                em_exercicios = False
                continue
            if em_exercicios:
                match = re.match(r'---- exercise \d+: (.+)', line)
                if match:
                    for question_id in match.group(1).split(' or '):
                        question_id = question_id.strip()
                        if question_id:
                            dados['question_ids'].append(question_id)
                continue
            match = re.match(r'---- (.+?): (.*)', line)
            if match:
                key, value = match.groups()
                if key in {'assessment title', 'class name', 'type'}:
                    dados[key.replace(' ', '_')] = value.strip()
    except (OSError, UnicodeError):
        return dados

    dados['question_ids'] = sorted(set(dados['question_ids']), key=lambda value: (not value.isdigit(), value))
    dados['path'] = filepath
    dados['filename'] = filepath.name
    return dados


def question_id_from_filename(filepath: Path) -> str | None:
    """Extrai o ID de questão dos nomes de executions, grades e codes.

    Execuções normalmente usam nomes como ``407_1326.log``; grades e
    códigos processados pela Etapa 1 usam frequentemente ``1326.log``.
    Ambos representam a questão 1326 e precisam ser indexados.
    """
    stem = filepath.stem
    if stem.isdigit():
        return stem
    if '_' not in stem:
        return None
    candidate = stem.rsplit('_', 1)[-1]
    return candidate if candidate.isdigit() else None



def extrair_codigo_ultimo_submition(execution_path: Path) -> str | None:
    """Extrai o código da última submissão oficial com GRADE.

    O arquivo execution.log é a fonte de verdade da versão efetivamente
    avaliada. Blocos TEST e SUBMITION anteriores são ignorados. A submissão
    oficial só é aceita quando possui GRADE e não possui ERROR.
    """
    separador = '*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*'
    try:
        conteudo = execution_path.read_text(encoding='utf-8', errors='replace')
    except (OSError, UnicodeError):
        return None
    blocos = [
        bloco for bloco in conteudo.split(separador)
        if re.search(r'^\s*==\s*SUBMITION\b', bloco, flags=re.IGNORECASE | re.MULTILINE)
    ]
    if not blocos:
        return None
    ultimo = blocos[-1]
    possui_erro = re.search(r'^\s*-+\s*ERROR\s*:', ultimo, flags=re.IGNORECASE | re.MULTILINE)
    possui_grade = re.search(r'^\s*-+\s*GRADE\s*:', ultimo, flags=re.IGNORECASE | re.MULTILINE)
    if possui_erro is not None or possui_grade is None:
        return None
    match = re.search(
        r'^\s*--\s*CODE\s*:\s*\n(.*?)(?=^\s*--\s*(?:EXECUTION TIME|TEST CASE|ERROR|GRADE)\s*:|\Z)',
        ultimo,
        flags=re.IGNORECASE | re.MULTILINE | re.DOTALL,
    )
    if match is None:
        return None
    codigo = match.group(1).strip('\n\r')
    return codigo if codigo.strip() else None


def validar_codigo_ultimo_submition(execution_path: Path) -> dict:
    """Valida a sintaxe do código do último SUBMITION com GRADE."""
    import ast
    resultado = {'path': str(execution_path), 'status': 'rejeitado', 'motivo': None, 'error_type': None, 'message': None, 'line': None, 'offset': None, 'code': None}
    codigo = extrair_codigo_ultimo_submition(execution_path)
    if codigo is None:
        resultado['motivo'] = 'ultimo_submition_sem_grade_ou_com_error_ou_sem_codigo'
        return resultado
    resultado['code'] = codigo
    try:
        ast.parse(codigo)
    except SyntaxError as exc:
        resultado.update({'motivo': 'codigo_do_ultimo_submition_com_sintaxe_invalida', 'error_type': type(exc).__name__, 'message': str(exc), 'line': exc.lineno, 'offset': exc.offset})
        return resultado
    resultado['status'] = 'aprovado'
    resultado['motivo'] = 'ultimo_submition_com_grade_e_sintaxe_valida'
    return resultado


def ultima_submissao_com_grade(execution_path: Path) -> bool:
    return validar_codigo_ultimo_submition(execution_path)['status'] == 'aprovado'

def verificar_execucao_candidata(
    tarefa: tuple[str, str, str, str, Path],
) -> dict:
    """Valida execution e sintaxe do código sem mutar estado compartilhado."""
    user_id, scope, assessment_id, question_id, filepath = tarefa
    resultado = validar_codigo_ultimo_submition(filepath)
    resultado.update({'id_usuario': user_id, 'scope': scope, 'assessment_id': assessment_id, 'question_id': question_id})
    if resultado['status'] == 'aprovado':
        resultado['reference'] = (user_id, scope, assessment_id, question_id)
    return resultado

def reference(filepath: Path, root: Path, question_id: str | None = None) -> dict[str, str]:
    item = {
        'path': filepath.relative_to(root).as_posix(),
        'name': filepath.name,
    }
    if question_id is not None:
        item['question'] = question_id
    return item


def escrever_relatorio_assessments_selecionados(assessments: list[dict], root: Path) -> Path:
    """Salva os assessments usados no recorte atual para auditoria/filtros."""
    relatorio = {
        'descricao': (
            'Assessments selecionados pela Etapa 2. O relatório permite revisar '
            'títulos como Matrizes e Trabalho Prático Substitutivo antes de aplicar '
            'um filtro adicional.'
        ),
        'filtros_aplicados': {
            'tipo': TIPO_ASSESSMENT_DESEJADO,
            'curso': NOME_CURSO_DESEJADO,
        },
        'quantidade_assessments': len(assessments),
        'group_definitions': GRUPOS_ASSESSMENT,
        'assessments': [
            {
                **assessment,
                'caminho_absoluto': str((root / assessment['path']).resolve()),
                'titulo_normalizado': normalizar(assessment.get('assessment_title', '')),
                'group_description': assessment.get('group_description'),
            }
            for assessment in sorted(assessments, key=lambda item: item.get('path', ''))
        ],
    }
    caminho = PASTA_SAIDA / 'assessments_selecionados_detalhado.json'
    caminho.parent.mkdir(parents=True, exist_ok=True)
    caminho.write_text(json.dumps(relatorio, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
    return caminho


GRUPOS_ASSESSMENT = {
    'TP 01': 'Variáveis e Estrutura Sequencial',
    'TP 02': 'Estrutura Condicional Simples e Composta',
    'TP 03': 'Estrutura Condicional Encadeada',
    'TP 04': 'Estrutura de Repetição por Condição',
    'TP 05': 'Vetores e Strings',
    'TP 06': 'Estrutura de Repetição por Contagem',
}

def classificar_grupo_assessment(titulo: str) -> str | None:
    """Classifica assessments nos grupos TP 01--06.

    A classificação usa o maior número TP encontrado no título. Assim,
    ``TP3 e TP4`` pertence ao grupo TP 04. Títulos sem grupo ou com termos
    explicitamente excluídos não entram no JSON.
    """
    texto = normalizar(titulo)
    texto_sem_acento = ''.join(
        caractere for caractere in unicodedata.normalize('NFD', texto)
        if unicodedata.category(caractere) != 'Mn'
    ).casefold()
    if not texto_sem_acento or texto_sem_acento == 'xxx':
        return None
    palavras_excluidas = (
        'matriz',
        'substitut',
        'prova final',
        'revisao',
    )
    if any(palavra in texto_sem_acento for palavra in palavras_excluidas):
        return None

    # Aceita TP, Trabalho Prático/Pratico e Avaliação/ Avaliacao numerados.
    padroes = (
        r'\btp\s*0*(\d+)\b',
        r'\btrabalho\s+pratic\w*\s*0*(\d+)\b',
        r'\bavaliac\w*\s+(?:parcial\s+)?0*(\d+)\b',
    )
    numeros = []
    for padrao in padroes:
        numeros.extend(int(valor) for valor in re.findall(padrao, texto_sem_acento))
    numeros_validos = [numero for numero in numeros if 1 <= numero <= 6]
    if not numeros_validos:
        return None
    numero = max(numeros_validos)
    return f'TP {numero:02d}'

def _scan_directory(path: Path) -> tuple[Path, list[Path], list[Path]]:
    """Lê uma pasta sem alterar estado compartilhado."""
    diretorios = []
    arquivos = []
    try:
        with os.scandir(path) as entries:
            for entry in entries:
                try:
                    if entry.is_dir(follow_symlinks=False):
                        if entry.name != 'mousemove':
                            diretorios.append(Path(entry.path))
                    elif entry.is_file(follow_symlinks=False):
                        arquivos.append(Path(entry.path))
                except OSError:
                    continue
    except OSError:
        return path, [], []
    return path, sorted(diretorios), sorted(arquivos)


def _parallel_walk(root: Path, workers: int):
    """Enumera diretórios em paralelo e produz resultados sem estado compartilhado.

    A função só paraleliza I/O de ``scandir``. A análise dos resultados continua
    no thread principal, preservando a mesma lógica e uma saída determinística.
    """
    from concurrent.futures import FIRST_COMPLETED, ThreadPoolExecutor, wait

    with ThreadPoolExecutor(max_workers=workers, thread_name_prefix='Etapa2-Scan') as executor:
        pendentes = {executor.submit(_scan_directory, root)}
        while pendentes:
            concluidos, pendentes = wait(pendentes, return_when=FIRST_COMPLETED)
            for futuro in sorted(concluidos, key=lambda item: str(item.result()[0])):
                caminho, diretorios, arquivos = futuro.result()
                yield caminho, diretorios, arquivos
                for diretorio in diretorios:
                    pendentes.add(executor.submit(_scan_directory, diretorio))

def build_references(extracted_root: Path, output_path: Path, tipo: str, nome_curso_filtro: str) -> dict:
    """Constrói o índice em O(E + F log F), com uma única varredura do disco.

    E é o número de entradas visitadas em Extraidos e F é o número de referências
    mantidas no JSON. O filtro de assessments é resolvido antes de incluir arquivos
    de usuário, sem copiar nenhum conteúdo.
    """
    root = extracted_root.resolve()
    assessments = []
    selected_questions: set[str] = set()
    selected_assessment_questions: set[tuple[str, str, str]] = set()
    selected_scope_questions: set[tuple[str, str]] = set()
    # O ID numérico do assessment pode se repetir entre datasets/turmas;
    # grades usam escopo + assessment. Questões continuam globais por ID.
    # não pelo ID de uma questão.
    selected_assessment_ids: set[tuple[str, str]] = set()
    # Mapa explícito usado para validar novamente cada grades/<assessment>.log.
    selected_assessment_types: dict[tuple[str, str], str] = {}
    tipo_norm = normalizar(tipo)
    nome_curso_norm = normalizar(nome_curso_filtro)
    user_files: list[tuple[str, str, Path, str | None, str | None, str]] = []
    users_roots: dict[str, set[str]] = {}
    execucao_candidates: list[tuple[str, str, str, str, Path]] = []
    execucoes_com_grade: set[tuple[str, str, str, str]] = set()
    falhas_execucoes: list[dict] = []
    ETAPA2_LOG_PATH.parent.mkdir(parents=True, exist_ok=True)
    ETAPA2_LOG_PATH.write_text('', encoding='utf-8')
    registrar_progresso('Etapa 2 iniciada')
    registrar_progresso(f'Configuração: {ETAPA2_WORKERS} workers para varredura e logs')
    registrar_progresso('Fase 1/4 iniciada — varredura paralela do disco')

    # Uma única caminhada coleta assessments e metadados dos usuários.
    diretorios_visitados = 0
    arquivos_encontrados = 0
    assessments_lidos = 0
    with tqdm(
        _parallel_walk(root, ETAPA2_WORKERS),
        desc=f'Fase 1/4 — varrendo disco ({ETAPA2_WORKERS} workers)',
        unit='diretório',
        dynamic_ncols=True,
        leave=True,
    ) as arvore:
        for current_path, dirnames_paths, filenames_paths in arvore:
            current_root = str(current_path)
            dirnames = [path.name for path in dirnames_paths]
            filenames = [path.name for path in filenames_paths]
            # Preserva as pastas mousemove no disco, mas não as percorre
            # nem inclui seu conteúdo no índice JSON.
            dirnames[:] = [nome for nome in dirnames if nome != 'mousemove']
            dirnames.sort()
            current = current_path
            diretorios_visitados += 1
            arquivos_encontrados += len(filenames)
            if diretorios_visitados == 1 or diretorios_visitados % 100 == 0:
                registrar_progresso(
                    f'Fase 1/4 | diretórios visitados={diretorios_visitados:,} | '
                    f'arquivos encontrados={arquivos_encontrados:,} | '
                    f'assessments lidos={assessments_lidos:,}'
                )
            relative_parts = current.relative_to(root).parts

            if current.name == 'assessments':
                for filename in sorted(filenames):
                    if not filename.endswith('.data'):
                        continue
                    filepath = current / filename
                    dados = ler_assessment_file(filepath)
                    assessments_lidos += 1
                    assessment_type = str(dados.get('type', ''))
                    class_name = str(dados.get('class_name', ''))
                    if tipo_norm and tipo_norm != 'todos' and normalizar(assessment_type) != tipo_norm:
                        continue
                    if nome_curso_norm != 'todos' and normalizar(class_name) != nome_curso_norm:
                        continue
                    grupo_assessment = classificar_grupo_assessment(
                        str(dados.get('assessment_title', ''))
                    )
                    if grupo_assessment is None:
                        continue
                    question_ids = sorted(
                        set(str(value) for value in dados['question_ids']),
                        key=lambda value: (not value.isdigit(), value),
                    )
                    selected_questions.update(question_ids)
                    scope = filepath.parent.parent.relative_to(root).as_posix()
                    selected_assessment_questions.update(
                        (scope, filepath.stem, question_id) for question_id in question_ids
                    )
                    selected_scope_questions.update(
                        (scope, question_id) for question_id in question_ids
                    )
                    # O arquivo 49.data corresponde a grades/49.log dentro do mesmo escopo.
                    selected_assessment_ids.add((scope, filepath.stem))
                    selected_assessment_types[(scope, filepath.stem)] = normalizar(assessment_type)
                    assessments.append({
                        'path': filepath.relative_to(root).as_posix(),
                        'filename': filepath.name,
                        'assessment_title': str(dados.get('assessment_title', '')),
                        'class_name': class_name,
                        'type': assessment_type,
                        'group': grupo_assessment,
                        'group_description': GRUPOS_ASSESSMENT[grupo_assessment],
                        'question_ids': question_ids,
                    })

            # A entrada users é identificada pelo caminho relativo, sem iniciar
            # uma segunda caminhada para cada usuário.
            if 'users' in relative_parts:
                users_index = relative_parts.index('users')
                if users_index + 1 < len(relative_parts):
                    user_id = relative_parts[users_index + 1]
                    user_dir = root.joinpath(*relative_parts[:users_index + 2])
                    users_roots.setdefault(user_id, set()).add(user_dir.relative_to(root).as_posix())
                    for filename in sorted(filenames):
                        filepath = current / filename
                        relative_to_user = filepath.relative_to(user_dir).parts
                        kind = relative_to_user[0] if relative_to_user else ''
                        if kind not in {'codes', 'executions', 'grades', 'codemirror'}:
                            if filename == 'user.data':
                                kind = 'user_data'
                            elif filename.startswith('final_grade'):
                                kind = 'final_grades'
                            elif filename == 'logins.log':
                                kind = 'logins'
                            else:
                                continue
                        question_id = question_id_from_filename(filepath)
                        relative_parts_user = filepath.relative_to(root).parts
                        users_index_user = relative_parts_user.index('users')
                        scope = '/'.join(relative_parts_user[:users_index_user])
                        assessment_id = (
                            filepath.stem.split('_', 1)[0]
                            if '_' in filepath.stem and filepath.stem.split('_', 1)[0].isdigit()
                            else None
                        )
                        user_files.append((user_id, kind, filepath, question_id, assessment_id, scope))
                        if (
                            kind == 'executions'
                            and question_id is not None
                            and assessment_id is not None
                        ):
                            execucao_candidates.append(
                                (user_id, scope, assessment_id, question_id, filepath)
                            )

    registrar_progresso(
        f'Fase 1/4 concluída — assessments={len(assessments):,}; '
        f'candidatos={len(execucao_candidates):,}'
    )
    registrar_progresso('Fase 2/4 iniciada — validação paralela dos execution logs')

    # Só agora, após conhecer os assessments/questões selecionados, lê os logs
    # candidatos. O trabalho é I/O-bound e é seguro paralelizar: as threads não
    # alteram o JSON nem qualquer coleção compartilhada.
    candidatos_validos = [
        tarefa for tarefa in execucao_candidates
        if (tarefa[1], tarefa[2]) in selected_assessment_ids
        and (tarefa[1], tarefa[2], tarefa[3]) in selected_assessment_questions
    ]
    logs_verificados = 0
    with ThreadPoolExecutor(max_workers=ETAPA2_WORKERS, thread_name_prefix='Etapa2-IO') as executor:
        resultados = executor.map(verificar_execucao_candidata, candidatos_validos)
        for resultado in tqdm(
            resultados,
            total=len(candidatos_validos),
            desc=f'Fase 2/4 — validando logs ({ETAPA2_WORKERS} workers)',
            unit='log',
            leave=True,
        ):
            logs_verificados += 1
            if logs_verificados == 1 or logs_verificados % 100 == 0:
                registrar_progresso(
                    f'Fase 2/4 | logs verificados={logs_verificados:,}/'
                    f'{len(candidatos_validos):,} | '
                    f'logs válidos={len(execucoes_com_grade):,}'
                )
            if resultado['status'] == 'aprovado':
                execucoes_com_grade.add(resultado['reference'])
            else:
                falha = dict(resultado)
                falha.pop('code', None)
                falhas_execucoes.append(falha)

    caminho_falhas_execucoes = PASTA_SAIDA / 'execucoes_rejeitadas_etapa2.json'
    caminho_falhas_execucoes.write_text(json.dumps({
        'schema_version': 1,
        'descricao': 'Executions rejeitados antes das etapas seguintes.',
        'criterio_aprovacao': 'último SUBMITION com GRADE, sem ERROR e sintaxe Python válida.',
        'total_rejeitados': len(falhas_execucoes),
        'falhas': falhas_execucoes,
    }, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
    registrar_progresso(f'Relatório de executions rejeitados salvo: {caminho_falhas_execucoes} | quantidade={len(falhas_execucoes):,}')

    registrar_progresso(
        f'Fase 2/4 concluída — logs válidos='
        f'{len(execucoes_com_grade):,}/{len(candidatos_validos):,}'
    )
    registrar_progresso('Fase 3/4 iniciada — montagem das referências')
    kinds = ('user_data', 'final_grades', 'codes', 'executions', 'grades', 'logins', 'codemirror')
    files_by_user: dict[str, dict[str, dict[str, dict[str, str]]]] = {}
    referencias_montadas = 0
    for user_id, kind, filepath, question_id, assessment_id, scope in user_files:
        # grades/49.log representa o assessment 49. Os demais arquivos
        # normalmente usam assessment_question (ex.: 407_1326.log); o filtro
        # precisa preservar somente pares pertencentes ao tipo selecionado.
        if kind == 'grades':
            chave_assessment = (scope, filepath.stem)
            # A existência do .log não basta: o .data correspondente precisa
            # ter sido selecionado pelo tipo solicitado (exam nesta execução).
            if chave_assessment not in selected_assessment_ids:
                continue
            tipo_grade = selected_assessment_types.get(chave_assessment)
            # Validação independente: abre o .data correspondente à própria
            # grade. Isso evita reutilizar seleção/JSON antigo ou aceitar uma
            # grade de homework por engano.
            caminho_assessment_grade = root / scope / 'assessments' / f'{filepath.stem}.data'
            metadados_grade = ler_assessment_file(caminho_assessment_grade)
            tipo_grade_direto = normalizar(str(metadados_grade.get('type', '')))
            if tipo_grade_direto != tipo_grade:
                continue
            if tipo_norm != 'todos' and tipo_grade_direto != tipo_norm:
                continue
        elif kind in {'codes', 'executions', 'codemirror'}:
            # O arquivo usa assessment_question (ex.: 798_1326.log ou
            # 798_996.py). A ocorrência só entra se o assessment pai estiver
            # explicitamente entre os assessments selecionados (exam por padrão).
            if question_id is None or assessment_id is None:
                continue
            if (scope, assessment_id) not in selected_assessment_ids:
                continue
            if (scope, assessment_id, question_id) not in selected_assessment_questions:
                continue
            if (user_id, scope, assessment_id, question_id) not in execucoes_com_grade:
                continue
        # Em grades, o número do arquivo é o assessment; não o ID da questão.
        item = reference(
            filepath,
            root,
            question_id if kind != 'grades' else None,
        )
        if kind == 'grades':
            item['assessment'] = filepath.stem
        files_by_user.setdefault(user_id, {}).setdefault(kind, {})[item['path']] = item
        referencias_montadas += 1
        if referencias_montadas == 1 or referencias_montadas % 1000 == 0:
            registrar_progresso(
                f'Fase 3/4 | referências montadas={referencias_montadas:,}/'
                f'{len(user_files):,}'
            )
    users = []
    for user_id in sorted(files_by_user):
        filtered_files = {
            kind: sorted(items.values(), key=lambda item: item['path'])
            for kind, items in files_by_user[user_id].items()
            if items
        }
        # A unidade de nota da Etapa 3 é grades/<assessment>.log. Portanto,
        # um execution de exam, sozinho, não prova que o usuário possui uma
        # nota final daquele assessment: a grade correspondente pode não existir.
        # Exemplo: executions/119_1118.log não substitui grades/119.log.
        # Só catalogamos o usuário quando existe pelo menos uma grade de um
        # assessment exam já selecionado; executions/codes/codemirror são dados
        # auxiliares das questões dessas grades.
        grades_exam_selecionadas = [
            item for item in filtered_files.get('grades', [])
            if selected_assessment_types.get(
                (
                    '/'.join(Path(item['path']).parts[:Path(item['path']).parts.index('users')]),
                    str(item.get('assessment') or Path(item['path']).stem),
                )
            ) == tipo_norm or tipo_norm == 'todos'
        ]
        if not grades_exam_selecionadas:
            continue
        # Mantém no JSON somente grades verificadas como pertencentes ao tipo
        # solicitado; nunca deixe homework entrar por uma referência antiga.
        filtered_files['grades'] = grades_exam_selecionadas
        # Exibe somente raízes que contêm arquivos mantidos pelo filtro.
        roots_selecionadas = set()
        for itens in filtered_files.values():
            for item in itens:
                partes = Path(item['path']).parts
                if 'users' in partes:
                    indice_users = partes.index('users')
                    if indice_users + 1 < len(partes):
                        roots_selecionadas.add('/'.join(partes[:indice_users + 2]))
        users.append({
            'id': user_id,
            'roots': sorted(roots_selecionadas),
            'files': filtered_files,
        })
        if len(users) == 1 or len(users) % 100 == 0:
            registrar_progresso(
                f'Fase 3/4 | usuários incluídos={len(users):,} | '
                f'referências montadas={referencias_montadas:,}'
            )
    # Invariante final do índice: antes de gravar o JSON, revalida cada
    # grade diretamente contra assessments/<id>.data. Esta barreira impede
    # que uma referência antiga, um ID coincidente ou uma execução de exam
    # sem grade faça um usuário chegar à Etapa 3.
    usuarios_rejeitados_sem_grade_tipo = 0
    usuarios_validados = []
    for registro in users:
        grades_validas = []
        for item in registro.get('files', {}).get('grades', []):
            path_grade = str(item.get('path', '')).replace('\\', '/')
            partes_grade = Path(path_grade).parts
            try:
                indice_users = partes_grade.index('users')
                escopo_grade = '/'.join(partes_grade[:indice_users])
            except ValueError:
                continue
            assessment_id_grade = str(item.get('assessment') or Path(path_grade).stem)
            caminho_assessment = root / escopo_grade / 'assessments' / f'{assessment_id_grade}.data'
            metadados = ler_assessment_file(caminho_assessment)
            tipo_real = normalizar(str(metadados.get('type', '')))
            if (caminho_assessment.is_file() and
                    (tipo_norm == 'todos' or tipo_real == tipo_norm)):
                grades_validas.append(item)
        if not grades_validas:
            usuarios_rejeitados_sem_grade_tipo += 1
            continue
        registro['files']['grades'] = grades_validas
        usuarios_validados.append(registro)
    users = usuarios_validados
    registrar_progresso(
        f'Validação final de grades: usuários aprovados={len(users):,}; '
        f'rejeitados sem grade do tipo={usuarios_rejeitados_sem_grade_tipo:,}'
    )
    registrar_progresso(
        f'Fase 3/4 concluída — usuários={len(users):,}; '
        f'arquivos={sum(len(v) for v in files_by_user.values()):,}'
    )
    registrar_progresso('Fase 4/4 iniciada — gravação do índice')

    source_root = os.path.relpath(root, output_path.resolve().parent).replace(os.sep, '/')
    statistics = {
        'assessments': len(assessments),
        'selected_questions': len(selected_questions),
        'users': len(users),
        'users_only_with_grade_of_selected_type': len(users),
        'files_by_kind': {
            kind: sum(len(user.get('files', {}).get(kind, [])) for user in users)
            for kind in kinds
        },
    }
    data = {
        'schema_version': 1,
        'description': 'Referências relativas aos arquivos filtrados em Extraidos; não contém cópias dos dados.',
        'source_root': source_root,
        'filters': {'tipo': tipo, 'curso': nome_curso_filtro, 'groups': list(GRUPOS_ASSESSMENT)},
        'group_definitions': GRUPOS_ASSESSMENT,
        'assessments': sorted(assessments, key=lambda item: item['path']),
        'users': users,
        'statistics': statistics,
    }
    output_path.parent.mkdir(parents=True, exist_ok=True)
    output_path.write_text(json.dumps(data, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
    # A quarta fase corresponde à gravação e confirmação da saída.
    registrar_progresso('Fase 4/4 concluída — saída gravada e validada')
    registrar_progresso(f'Índice salvo: {output_path.resolve()}')
    registrar_progresso(f'Log completo salvo: {ETAPA2_LOG_PATH.resolve()}')
    return data


referencias = build_references(PASTA_EXTRAIDOS, CAMINHO_REFERENCIAS, TIPO_ASSESSMENT_DESEJADO, NOME_CURSO_DESEJADO)
CAMINHO_ASSESSMENTS_DETALHADO = escrever_relatorio_assessments_selecionados(
    referencias['assessments'], Path(PASTA_EXTRAIDOS)
)
print(f'Referências criadas: {CAMINHO_REFERENCIAS.resolve()}')
print(f'Relatório de assessments selecionados: {CAMINHO_ASSESSMENTS_DETALHADO.resolve()}')
print(f"Assessments filtrados: {referencias['statistics']['assessments']}")
print(f"Questões selecionadas: {referencias['statistics']['selected_questions']}")
print(f"Usuários incluídos: {referencias['statistics']['users']}")
print(f"Arquivos referenciados: {sum(referencias['statistics']['files_by_kind'].values())}")


[2026-11-06 21:13:07] Etapa 2 iniciada


[2026-11-06 21:13:07] Configuração: 16 workers para varredura e logs


[2026-11-06 21:13:07] Fase 1/4 iniciada — varredura paralela do disco


Fase 1/4 — varrendo disco (16 workers): 0diretório [00:00, ?diretório/s]

[2026-11-06 21:13:07] Fase 1/4 | diretórios visitados=1 | arquivos encontrados=0 | assessments lidos=0


Fase 1/4 — varrendo disco (16 workers): 30diretório [00:00, 91.01diretório/s]

... (4,498 outputs left out)

[2026-11-06 21:27:05] Fase 4/4 concluída — saída gravada e validada


[2026-11-06 21:27:05] Índice salvo: D:\Trabalho_Super\QuestionInsight\Etapa_2\output\referencias_processamento.json


[2026-11-06 21:27:05] Log completo salvo: D:\Trabalho_Super\QuestionInsight\Etapa_2\output\etapa2_progresso.log


Referências criadas: D:\Trabalho_Super\QuestionInsight\Etapa_2\output\referencias_processamento.json
Relatório de assessments selecionados: D:\Trabalho_Super\QuestionInsight\Etapa_2\output\assessments_selecionados_detalhado.json
Assessments filtrados: 611
Questões selecionadas: 1187
Usuários incluídos: 2979
Arquivos referenciados: 117508


## 3. Barreira final por grade e saída do índice

A unidade de nota usada pela Etapa 3 é `grades/<assessment>.log`. Por isso, um execution de exam isolado não é suficiente para catalogar um usuário: é obrigatório existir pelo menos uma grade cujo assessment correspondente seja realmente do tipo selecionado. Antes de gravar o JSON, cada grade é novamente conferida contra `assessments/<id>.data`.

A saída `referencias_processamento.json` contém os assessments selecionados, os usuários aprovados, os arquivos filtrados, as definições dos grupos e estatísticas do recorte. A validação final garante que todo usuário listado possua ao menos uma grade válida do tipo escolhido.


In [4]:
import json

referencias = json.loads(CAMINHO_REFERENCIAS.read_text(encoding='utf-8'))
assert referencias['schema_version'] == 1
assert 'source_root' in referencias
assert all('path' in item for item in referencias.get('assessments', []))
assert all('id' in user and 'files' in user for user in referencias.get('users', []))
print('Validação concluída: JSON estruturado e sem arquivos copiados pela Etapa 2.')


Validação concluída: JSON estruturado e sem arquivos copiados pela Etapa 2.
